## Tutorial: CommonRoad Curvilinear Coordinatesystem

This tutorial shows you how to create a curvilinear coordinate system using CommonRoad scenarios. We start with opening a CommonRoad XML file.

In [ ]:
%matplotlib inline
import os
import matplotlib.pyplot as plt
from commonroad.common.file_reader import CommonRoadFileReader
from commonroad.visualization.draw_dispatch_cr import draw_object

# load the CommonRoad scenario, note that you might have to modify the path to the CommonRoad scenario!
file_path = os.path.join(os.getcwd(), '../../commonroad/scenarios/NGSIM/US101/USA_US101-1_1_T-1.xml')

scenario, planning_problem_set = CommonRoadFileReader(file_path).open()

# plot the scenario for each time step
for i in range(0, 5):
    plt.figure(figsize=(25, 10))
    draw_object(scenario, draw_params={'time_begin': i, 'scenario': {'lanelet_network': {
                            'lanelet': {'show_label': True}}}})
    draw_object(planning_problem_set)
    plt.gca().set_aspect('equal')
    plt.show()

We use the center line of the lanelet with ID 536 as reference path for our curvilinear coordinatesystem.

In [ ]:
from commonroad_ccosy.geometry.trapezoid_coordinate_system import create_coordinate_system_from_polyline

reference_path = scenario.lanelet_network.find_lanelet_by_id(536).center_vertices
curvilinear_cosy = create_coordinate_system_from_polyline(reference_path)

We can visualize the segments of the coordinate system with the commonroad_ccosy.visualization module. 

In [ ]:
import commonroad_ccosy.visualization.draw_dispatch

plt.figure(figsize=(25, 10))
draw_object(scenario, draw_params={'time_begin': i, 'scenario': {'lanelet_network': {
                        'lanelet': {'show_label': True}}}})
commonroad_ccosy.visualization.draw_dispatch.draw_object(curvilinear_cosy.get_segment_list())
draw_object(planning_problem_set)
plt.gca().set_aspect('equal')
plt.show()

As can be seen in the picture above, the reference path is jagged in the beginning of the lanelet. We can smooth the reference path with the Chaikins corner cutting algorithm followed by a respampling of the reference path. Here, we resample every 2.5m. 

In [ ]:
from commonroad_ccosy.geometry.util import chaikins_corner_cutting, resample_polyline

# we smooth the reference path 10 times
for i in range(0, 10):
    reference_path = chaikins_corner_cutting(reference_path)
reference_path = resample_polyline(reference_path, 2.5)

Now, we create again a new coordinate system with the new reference path.

In [ ]:
curvilinear_cosy_new = create_coordinate_system_from_polyline(reference_path)

plt.figure(figsize=(25, 10))
draw_object(scenario, draw_params={'time_begin': i, 'scenario': {'lanelet_network': {
                        'lanelet': {'show_label': True}}}})
commonroad_ccosy.visualization.draw_dispatch.draw_object(curvilinear_cosy_new.get_segment_list())
draw_object(planning_problem_set)
plt.gca().set_aspect('equal')
plt.show()

Now, we can convert coordinates from the Cartesian frame to the curvilinear coordinate system and backwards.

In [ ]:
import numpy as np

p_cartesian = np.array([0, 0])

p_curvilinear = curvilinear_cosy_new.convert_to_curvilinear_coords(p_cartesian[0], p_cartesian[1])
print('Converted p_cartesian: {}'.format(p_curvilinear))

p_cartesian = curvilinear_cosy_new.convert_to_cartesian_coords(p_curvilinear[0], p_curvilinear[1])
print('Converted p_curvilinear: {}'.format(p_cartesian))
